# 1. Librairies importation

In [3]:
%pip install -r ../requirements.txt

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import GroupKFold

Note: you may need to restart the kernel to use updated packages.


# 2. Adapating our columns


In [11]:
import pandas as pd
df_reg = pd.read_csv("/Users/erin.so/Documents/Cheminformatics/Data-Project-for-Cheminformatics-class/data/processed/solubility_clean.csv")


# 3. Bemis-Murcko scaffolds to separate molecules with the same scaffold in training and test

In [12]:
def get_scaffold(smiles): ## NEEDs to call SCAFFOLD FUNCTION FROM EVALUATION.PY 
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    scaffold = Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
    return scaffold if scaffold else smiles


smiles_col = "Canonical_smiles"
df_reg["scaffold"] = df_reg[smiles_col].apply(get_scaffold)

invalid_count = df_reg["scaffold"].isna().sum()
print("Invalid SMILES:", invalid_count)

# Remove molecules with invalid structures
df_reg = df_reg.dropna(subset=["scaffold"]).copy()

print("Valid molecules:", len(df_reg))
print("Unique scaffolds:", df_reg["scaffold"].nunique())


df_reg["scaffold"] = df_reg[smiles_col].apply(get_scaffold)


invalid_count = df_reg["scaffold"].isna().sum()
print("Invalid SMILES:", invalid_count)

# Remove molecules with invalid structures
df_reg = df_reg.dropna(subset=["scaffold"]).copy()

print("Valid molecules:", len(df_reg))
print("Unique scaffolds:", df_reg["scaffold"].nunique())

[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:23] WARNING: not removing hydrogen atom without neighbors
[23:43:24] WARNING: not removing hydrogen atom without neighbors
[23:43:24] WARNING: not removing hydrogen atom without neighbors
[23:43:24] WARNING: not removing hydrogen atom without neighbors
[23:43:24] WARNING: not removing hydrogen atom without neighbors
[23:43:24] WARNING: not r

Invalid SMILES: 0
Valid molecules: 9982
Unique scaffolds: 4996


[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:31] WARNING: not removing hydrogen atom without neighbors
[23:43:32] WARNING: not removing hydrogen atom without neighbors
[23:43:32] WARNING: not removing hydrogen atom without neighbors
[23:43:32] WARNING: not removing hydrogen atom without neighbors
[23:43:32] WARNING: not r

Invalid SMILES: 0
Valid molecules: 9982
Unique scaffolds: 4996


# 4. Test-split 

In [13]:
# Size of each scaffold group
scaffold_sizes = df_reg["scaffold"].value_counts()

print(scaffold_sizes.describe())
print("Largest scaffold group:", scaffold_sizes.max())

# Splitting our molecules by scafold 
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df_reg, groups=df_reg["scaffold"])
)

train_df = df_reg.iloc[train_idx].copy()
test_df = df_reg.iloc[test_idx].copy()

print("Training set:", len(train_df), "molecules")
print("Test set:", len(test_df), "molecules")
print("Test proportion:", round(len(test_df) / len(df_reg) * 100, 2), "%")


count    4996.000000
mean        1.997998
std        25.393243
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max      1753.000000
Name: count, dtype: float64
Largest scaffold group: 1753
Training set: 8354 molecules
Test set: 1628 molecules
Test proportion: 16.31 %


# 5. Freeze the split from test-split 

In [14]:
df_reg["split"] = "train"
df_reg.loc[test_df.index, "split"] = "test"

df_reg.to_csv("saved_scaffold_split.csv", index=False)

print(df_reg["split"].value_counts())
print("Split assignments saved.")

split
train    8354
test     1628
Name: count, dtype: int64
Split assignments saved.
